# 04 — Geç füzyon (E4, E4b) ve karşılaştırmalar (CPU)
**Çalışma zamanı → CPU** (GPU gerekmez). Secrets: `KAGGLE_API_TOKEN`, `GITHUB_TOKEN`.
Gerekenler Drive'da: E1 ve E2'nin değerlendirme tahminleri (`eval/`); sağlamlık için
03_robust'ın çıktıları (`robust/`, E2 hizasızlığı dahil); karşılaştırmalar için E3 ve E3c.
**Kullanım:** ilk hücrede anahtarları ayarla → **Çalışma zamanı → Tümünü çalıştır**.
Biten adımlar atlanır; oturum koparsa yeniden Tümünü çalıştır.

In [ ]:
BRANCH = 'h4'
SELECT = True    # 1) E4 ve E4b parametre seçimi (sadece val) + run JSON'ları
ROBUST = True    # 2) geç füzyonun sağlamlık paketi (03_robust bitmiş olmalı)
COMPARE = True   # 3) eşleştirilmiş karşılaştırmalar (temiz + kamera kaybı)
DRIVE = '/content/drive/MyDrive/baykus'
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
from google.colab import userdata
os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')
print('token yüklendi (değer yazdırılmadı)')

In [ ]:
# Repo private ise Secrets'a GITHUB_TOKEN ekle (fine-grained, sadece bu repo, 'Contents: read').
import os
from google.colab import userdata
try:
    tok = userdata.get('GITHUB_TOKEN')
except Exception:
    tok = None
url = f'https://{tok}@github.com/OgzhnTarhn/baykus.git' if tok else 'https://github.com/OgzhnTarhn/baykus.git'
if not os.path.isdir('/content/baykus/.git'):
    !git clone -q {url} /content/baykus
!git -C /content/baykus fetch -q {url} {BRANCH}
!git -C /content/baykus checkout -q -B {BRANCH} FETCH_HEAD
!git -C /content/baykus remote set-url origin https://github.com/OgzhnTarhn/baykus.git
%cd /content/baykus
!git log --oneline -3

In [ ]:
%%time
!pip -q install uv kaggle
!nproc   # işçi sayısı: birleştirme süresini belirler
!uv pip install --system -q -e /content/baykus/python
import sys
sys.path.insert(0, '/content/baykus/python/src')

In [ ]:
%%time
# İşlenmiş veri: private Kaggle veri seti (ADR 0004). Kaggle yüklenen tar'ları klasöre açtı;
# CLI klasör seçerek indiremediği için veri setinin tamamı iner (~9,8 GB).
DS = 'ouzhantarhan/baykus-dv-rgbt'
!kaggle datasets download {DS} -p /content/dv_rgbt --unzip -q
!mv /content/dv_rgbt/meta /content/meta
!baykus-data illum-relabel --meta /content/meta   # ADR 0005: Kaggle'daki etiketler eski eşikle
!baykus-data slices --out /content/dv_rgbt --meta /content/meta
!baykus-data yaml --out /content/dv_rgbt
!du -sh /content/dv_rgbt/* /content/meta

In [ ]:
# Modeller: her deneyin biten en son tam eğitimi → run_id
import json, os
from pathlib import Path
from baykus.train.checkpoints import find_runs
NPROC = os.cpu_count()

def train_run(exp):
    done = find_runs(f'{DRIVE}/runs', exp, finished=True)
    return json.loads((done[-1] / 'baykus_train.json').read_text(encoding='utf-8'))['run_id'] if done else None

RUN = {e: train_run(e) for e in ['e1_rgb', 'e2_ir', 'e3_early_fusion', 'e3c_modality_dropout']}
for e, r in RUN.items():
    print(e, '→', r)
assert RUN['e1_rgb'] and RUN['e2_ir'], 'E1 ve E2 biten eğitim yok'
EVAL = lambda r: f'{DRIVE}/eval/{r}_val/predictions_coco.json.gz'
SEL = lambda exp: f'{DRIVE}/results/fusion/{exp}_val_selection.json'

In [ ]:
%%time
# 1) E4: sabit ağırlık (12 hücreli ızgara); E4b: aydınlatmaya göre ağırlık (E4'ün IoU eşiğiyle).
# Süreyi ve 'map50' satırlarını Claude'a getir.
COMMON = (f'--rgb-preds {EVAL(RUN["e1_rgb"])} --ir-preds {EVAL(RUN["e2_ir"])} '
          f'--rgb-run-id {RUN["e1_rgb"]} --ir-run-id {RUN["e2_ir"]} '
          f'--data-root /content/dv_rgbt --meta /content/meta '
          f'--project {DRIVE}/eval --results {DRIVE}/results --workers {NPROC}')
if SELECT:
    !cd /content/baykus/python && baykus-fusion select --config ../configs/fusion/e4_late_fusion.yaml {COMMON}
    !cd /content/baykus/python && baykus-fusion select --config ../configs/fusion/e4b_late_fusion_illum.yaml --base {SEL('e4_late_fusion')} {COMMON}
else:
    print('SELECT = False → atlandı')

In [ ]:
%%time
# 2) Sağlamlık: E1 ve E2'nin her koşuldaki tahminleri birleştirilir (GPU yok). Biten koşul atlanır.
if ROBUST:
    for EXP in ['e4_late_fusion', 'e4b_late_fusion_illum']:
        !cd /content/baykus/python && baykus-fusion robust --config ../configs/fusion/{EXP}.yaml --selection {SEL(EXP)} --robust-preds {DRIVE}/robust --data-root /content/dv_rgbt --meta /content/meta --project {DRIVE}/robust --results {DRIVE}/results --workers {NPROC}
else:
    print('ROBUST = False → atlandı')

In [ ]:
# 3) Eşleştirilmiş karşılaştırmalar (val, IR etiketleri; FR-V8). Eksik model/koşul atlanır.
for EXP in ['e4_late_fusion', 'e4b_late_fusion_illum']:
    p = Path(SEL(EXP))
    RUN[EXP] = json.loads(p.read_text(encoding='utf-8'))['run_id'] if p.exists() else None

def preds(exp, cond):
    r = RUN.get(exp)
    if r is None:
        return None
    p = Path(EVAL(r) if cond == 'clean' else f'{DRIVE}/robust/{r}_val_{cond}/predictions_coco.json.gz')
    return p if p.exists() else None

PAIRS = [('e3c_modality_dropout', 'e3_early_fusion', 'clean'),
         ('e4_late_fusion', 'e3_early_fusion', 'clean'),
         ('e4_late_fusion', 'e2_ir', 'clean'),
         ('e4b_late_fusion_illum', 'e4_late_fusion', 'clean')]
PAIRS += [(a, b, c) for c in ('drop_rgb', 'drop_ir') for a, b in [
    ('e3c_modality_dropout', 'e3_early_fusion'),
    ('e4_late_fusion', 'e3_early_fusion'),
    ('e4_late_fusion', 'e3c_modality_dropout')]]
# Kamera kaybı puanı (seçim kuralı): iki kayıp koşulunun ortalaması, birleşik bootstrap
CAMLOSS = [('e4_late_fusion', 'e3c_modality_dropout'),
           ('e3c_modality_dropout', 'e3_early_fusion'),
           ('e4_late_fusion', 'e3_early_fusion')]
if COMPARE:
    for A, B in CAMLOSS:
        name = f'{A}-vs-{B}_val_camloss_ir'
        if Path(f'{DRIVE}/results/compare/{name}.json').exists():
            print('zaten var:', name)
            continue
        PA = [preds(A, c) for c in ('drop_rgb', 'drop_ir')]
        PB = [preds(B, c) for c in ('drop_rgb', 'drop_ir')]
        if None in PA + PB:
            print('atlandı (tahmin yok):', name)
            continue
        !cd /content/baykus/python && baykus-eval compare --gt /content/dv_rgbt/coco/val_ir.json --a "{PA[0]}" "{PA[1]}" --b "{PB[0]}" "{PB[1]}" --slices /content/meta/slices_val.json --split val --name {name} --results {DRIVE}/results
    for A, B, C in PAIRS:
        name = f'{A}-vs-{B}_val_ir' if C == 'clean' else f'{A}-vs-{B}_val_{C}_ir'
        if Path(f'{DRIVE}/results/compare/{name}.json').exists():
            print('zaten var:', name)
            continue
        PA, PB = preds(A, C), preds(B, C)
        if PA is None or PB is None:
            print('atlandı (tahmin yok):', name)
            continue
        !cd /content/baykus/python && baykus-eval compare --gt /content/dv_rgbt/coco/val_ir.json --a "{PA}" --b "{PB}" --slices /content/meta/slices_val.json --split val --name {name} --results {DRIVE}/results
else:
    print('COMPARE = False → atlandı')